In [2]:
import pandas as pd

# 读取 parquet 文件
df = pd.read_parquet("ar-00000-of-00001.parquet", engine="pyarrow")  # 或 engine="fastparquet"

# 保存为 CSV
df.to_csv("ar-00000-of-00001.csv", index=False)

print("转换完成，已保存为 ar-00000-of-00001.csv")

转换完成，已保存为 ar-00000-of-00001.csv


In [1]:
import pandas as pd
import csv

# ========== 去重配置 ==========
DEDUP_KEEP = 'first'   # 'first' 或 'last'

# ========== 文件配置列表 ==========
file_configs = [
    # 示例：单列标签，除了 "normal" 外都是有毒
    # {
    #     "path": "ar_dataset.csv",
    #     "sep": ",",
    #     "tweet_column": "tweet",            # 推文所在列
    #     "label_column": "sentiment",           # 标签列名（包含 normal, hate, offensive 等）
    #     "label_normal_value": "normal",    # 代表无毒的标签值，其余全为有毒(1)
    #     "header": 0
    # },
    # {
    #     "path": "let-mi_train_part.csv",
    #     "sep": ",",
    #     "tweet_column": "text",            # 推文所在列
    #     "label_column": "category",           # 标签列名（包含 normal, hate, offensive 等）
    #     "label_normal_value": "none",    # 代表无毒的标签值，其余全为有毒(1)
    #     "header": 0
    # },
    #     {
    #     "path": "L-HSAB.csv",
    #     "sep": ",",
    #     "tweet_column": "Tweet",
    #     "label_column": "Class",
    #     "header": 0
    # },
    # {
    #     "path": "tr.csv",
    #     "sep": ",",
    #     "tweet_column": "text",
    #     "toxicity_columns": [
    #         "Hate_Speech",
    #         "Targeted_Harassment",
    #         "NSFW_Sexual",
    #         "Violence_Incitement",
    #         "Dangerous_Ideology",
    #         "Profanity_Slang"
    #     ],
    #     "header": 0
    # },
    {
        "path": "pt.csv",
        "sep": ",",
        "tweet_column": "text",
        "label_column": "label",
        "header": 0
    },
    {
        "path": "pt_1.csv",
        "sep": ",",
        "tweet_column": "text",
        "label_column": "class_clean",
        "header": 0
    },
    # {
    #     "path": "tr_hf.csv",
    #     "sep": ",",
    #     "tweet_column": "text",
    #     "label_column": "labels",
    #     "header": 0
    # },
    # {
        
    #     "path": "train.csv",
    #     "sep": ",",
    #     "tweet_column": "text",
    #     "label_column": "label",
    #     "header": 0
    # },
    # {
    #     "path": "ar-00000-of-00001.csv",
    #     "sep": ",",
    #     "tweet_column": "text",
    #     "label_column": "toxic",
    #     "header": 0
    # },
    # {
    #     "path": "offenseval-ar-training-v1.tsv",
    #     "sep": "\t",
    #     "tweet_column": "tweet",
    #     "label_column": "subtask_a",
    #     "label_normal_value": "NOT",    # 代表无毒的标签值，其余全为有毒(1)
    #     "header": 0
    # },
    # 可以继续添加其他文件（支持多种模式混合）
    # {
    #     "path": "multi_label_file.csv",
    #     "tweet_column": "Tweet",
    #     "toxicity_columns": ["Hate_Speech", "Targeted_Harassment"],
    #     "header": 0
    # },
]

# ========== 工具函数 ==========
def read_file_with_fallback(file_config):
    """多编码读取CSV/TXT"""
    file_path = file_config["path"]
    sep = file_config.get("sep", ",")
    header = file_config.get("header", 0)
    encoding_list = file_config.get("encoding_try", ['utf-8', 'utf-8-sig', 'latin-1', 'cp1252'])
    for enc in encoding_list:
        try:
            df = pd.read_csv(file_path, sep=sep, header=header, encoding=enc)
            print(f"  ✓ 使用编码 {enc} 成功读取 {file_path}")
            return df
        except UnicodeDecodeError:
            continue
        except Exception as e:
            print(f"  ✗ 使用编码 {enc} 读取失败: {e}")
            continue
    print(f"  ⚠ 所有编码均失败，使用 latin-1 强制读取")
    return pd.read_csv(file_path, sep=sep, header=header, encoding='latin-1')

def extract_toxicity_label(df, config):
    """返回二值标签 Series (0/1/-1)"""
    # 模式1：多列二值合并
    if "toxicity_columns" in config:
        cols = config["toxicity_columns"]
        existing = [c for c in cols if c in df.columns]
        if not existing:
            print(f"  ⚠ 未找到任何 toxicity_columns 列: {cols}")
            return pd.Series([-1] * len(df))
        label_series = df[existing].max(axis=1).fillna(0).astype(int)
        print(f"  ✓ 使用多列合并 (OR) 生成毒性标签，涉及列: {existing}")
        return label_series

    # 模式2：单列标签
    if "label_column" in config:
        col = config["label_column"]
        if col not in df.columns:
            print(f"  ✗ 未找到 label_column: {col}")
            return pd.Series([-1] * len(df))

        raw = df[col]

        # 新增：使用 label_normal_value（除了指定值，其他全为1）
        if "label_normal_value" in config:
            normal_vals = config["label_normal_value"]
            if isinstance(normal_vals, str):
                normal_vals = [normal_vals]
            # 默认全为1（有毒）
            label_series = pd.Series(1, index=df.index)
            # 正常值的位置设为0
            mask_normal = raw.isin(normal_vals)
            label_series[mask_normal] = 0
            # NaN 不会被 isin 匹配，保持为1
            print(f"  ✓ 使用 label_normal_value={normal_vals} 转换：这些值 -> 0，其他所有值（包括缺失）-> 1")
            return label_series

        # 使用映射字典
        mapping = config.get("label_mapping")
        if mapping is not None:
            label_series = raw.map(mapping)
            if label_series.isna().any():
                unmapped = raw[label_series.isna()].unique()
                print(f"  ⚠ 发现未映射的值: {unmapped}，将填充为-1")
                label_series = label_series.fillna(-1).astype(int)
            else:
                label_series = label_series.astype(int)
            print(f"  ✓ 使用映射字典转换标签列 '{col}'")
            return label_series
        else:
            # 无映射：尝试直接转为int
            try:
                label_series = raw.astype(int)
                if label_series.isin([0,1]).all():
                    print(f"  ✓ 标签列 '{col}' 已是0/1整数，直接使用")
                else:
                    print(f"  ⚠ 标签列 '{col}' 包含非0/1的整数，将保留原值")
                return label_series
            except (ValueError, TypeError):
                print(f"  ✗ 标签列 '{col}' 无法转换为整数，且未提供映射或 normal_value → 标签全为-1")
                return pd.Series([-1] * len(df))

    # 两者都没有
    print(f"  ⚠ 配置中既没有 toxicity_columns 也没有 label_column → 标签全为-1")
    return pd.Series([-1] * len(df))

def extract_tweets_and_labels(df, config):
    """提取推文和标签，返回(tweets列表, labels列表)"""
    tweet_col = config["tweet_column"]
    if tweet_col not in df.columns:
        print(f"  ✗ 未找到推文列 '{tweet_col}'，实际列名: {df.columns.tolist()}")
        return [], []

    non_empty = df[tweet_col].notna()
    valid_df = df[non_empty].copy()
    tweets = valid_df[tweet_col].tolist()

    full_labels = extract_toxicity_label(df, config)
    labels = full_labels[non_empty].tolist()
    labels = [str(l) for l in labels]   # 转为字符串保存

    valid_01_count = sum(1 for l in labels if l in ('0','1'))
    print(f"  提取到 {len(tweets)} 条推文，其中有效标签（0/1）数: {valid_01_count}")
    return tweets, labels

def main():
    all_tweets = []
    all_labels = []

    for idx, config in enumerate(file_configs, 1):
        print(f"\n处理文件 {idx}: {config['path']}")
        df = read_file_with_fallback(config)
        tweets, labels = extract_tweets_and_labels(df, config)
        all_tweets.extend(tweets)
        all_labels.extend(labels)

    raw_df = pd.DataFrame({'tweet': all_tweets, 'label': all_labels})
    raw_output = "all_datasets_with_labels.csv"
    raw_df.to_csv(raw_output, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)
    print(f"\n✓ 原始合并: {len(raw_df)} 条记录，已保存至 {raw_output}")

    dedup_df = raw_df.drop_duplicates(subset=['tweet'], keep=DEDUP_KEEP)
    removed = len(raw_df) - len(dedup_df)
    print(f"✓ 去重后: {len(dedup_df)} 条记录（移除 {removed} 条重复）")

    dedup_output = "all_datasets_with_labels_dedup.csv"
    dedup_df.to_csv(dedup_output, index=False, encoding='utf-8', quoting=csv.QUOTE_MINIMAL)
    print(f"✓ 去重结果已保存至 {dedup_output}")

    print("\n去重后前5条预览：")
    print(dedup_df.head())

if __name__ == "__main__":
    main()


处理文件 1: pt.csv
  ✓ 使用编码 utf-8 成功读取 pt.csv
  ✓ 标签列 'label' 已是0/1整数，直接使用
  提取到 35838 条推文，其中有效标签（0/1）数: 35838

处理文件 2: pt_1.csv
  ✓ 使用编码 utf-8 成功读取 pt_1.csv
  ⚠ 标签列 'class_clean' 包含非0/1的整数，将保留原值
  提取到 45000 条推文，其中有效标签（0/1）数: 44903

✓ 原始合并: 80838 条记录，已保存至 all_datasets_with_labels.csv
✓ 去重后: 78867 条记录（移除 1971 条重复）
✓ 去重结果已保存至 all_datasets_with_labels_dedup.csv

去重后前5条预览：
                                               tweet label
0     Bem-vindo(a), LUCIANE MOLINA MORAES!    Seu...     0
1  Só há uma coisa que eu detesto mais que um atr...     0
2                             nem tudo  q parece 茅馃槈     0
3  Do UOL, em São Paulo, e colaboração para o UOL...     0
4  25px Bem-vindo(a) à Wikipédia. Todas as pessoa...     0
